# 2D heat equation on the GPU (CUDA)

1. **Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works).
2. **Runtime → Run all**.

The notebook writes the project files, builds them with CMake, checks every CUDA kernel against a CPU run,
benchmarks them, and plots the temperature. The tables are saved to `results/results_cuda.md`.

In [ ]:
# Project files (generated by colab/make_notebook.py from the repository)
import os

FILES = {
    'CMakeLists.txt': 'cmake_minimum_required(VERSION 3.16)\nproject(heat2d LANGUAGES CXX)\n\nset(CMAKE_CXX_STANDARD 17)\nset(CMAKE_CXX_STANDARD_REQUIRED ON)\nset(CMAKE_EXPORT_COMPILE_COMMANDS ON)\n\nif(NOT CMAKE_BUILD_TYPE)\n  set(CMAKE_BUILD_TYPE Release)\nendif()\n\nfind_package(MPI REQUIRED)\nadd_executable(heat src/mpi/main.cpp src/mpi/heat_mpi.cpp)\ntarget_compile_options(heat PRIVATE -Wall -Wextra)\ntarget_link_libraries(heat PRIVATE MPI::MPI_CXX)\n\nset(NP 4 CACHE STRING "Number of MPI processes")\nset(HEAT_K 1000 CACHE STRING "Time steps")\nset(HEAT_N 256 CACHE STRING "Grid size")\n\nadd_custom_target(run\n  COMMAND ${MPIEXEC_EXECUTABLE} ${MPIEXEC_NUMPROC_FLAG} ${NP} $<TARGET_FILE:heat> ${HEAT_K} ${HEAT_N}\n  DEPENDS heat\n  USES_TERMINAL\n)\n\n# Optional CUDA build: only when a CUDA compiler (nvcc) is found.\n#   heat_cuda      single GPU (kernels: naive, shared, temporal; double or float)\n#   heat_mpi_cuda  MPI + CUDA, one GPU per rank\ninclude(CheckLanguage)\ncheck_language(CUDA)\nif(CMAKE_CUDA_COMPILER)\n  # Compile for the GPU in this machine unless an architecture is given (e.g. -DCMAKE_CUDA_ARCHITECTURES=75).\n  if(NOT DEFINED CMAKE_CUDA_ARCHITECTURES AND CMAKE_VERSION VERSION_GREATER_EQUAL 3.24)\n    set(CMAKE_CUDA_ARCHITECTURES native)\n  endif()\n  set(CMAKE_CUDA_RUNTIME_LIBRARY Shared)\n  enable_language(CUDA)\n  find_package(CUDAToolkit REQUIRED)\n  set(CMAKE_CUDA_STANDARD 17)\n  set(CMAKE_CUDA_STANDARD_REQUIRED ON)\n\n  # Kernels; the programs that use them are plain C++.\n  add_library(heat_gpu STATIC src/cuda/heat_gpu.cu)\n  target_compile_options(heat_gpu PRIVATE -lineinfo)   # source lines in Nsight Compute profiles\n  target_link_libraries(heat_gpu PUBLIC CUDA::cudart)\n\n  add_executable(heat_cuda src/cuda/heat_cuda.cpp)\n  target_compile_options(heat_cuda PRIVATE -Wall -Wextra)\n  target_link_libraries(heat_cuda PRIVATE heat_gpu)\n\n  add_executable(heat_mpi_cuda src/cuda/heat_mpi_cuda.cpp)\n  target_compile_options(heat_mpi_cuda PRIVATE -Wall -Wextra)\n  target_link_libraries(heat_mpi_cuda PRIVATE heat_gpu MPI::MPI_CXX)\n\n  set(HEAT_CUDA_KERNEL naive CACHE STRING "Kernel for the run_cuda target: naive, shared or temporal")\n  add_custom_target(run_cuda\n    COMMAND $<TARGET_FILE:heat_cuda> ${HEAT_K} ${HEAT_N} --kernel ${HEAT_CUDA_KERNEL}\n    DEPENDS heat_cuda\n    USES_TERMINAL\n  )\n  add_custom_target(run_mpi_cuda\n    COMMAND ${MPIEXEC_EXECUTABLE} ${MPIEXEC_NUMPROC_FLAG} ${NP} $<TARGET_FILE:heat_mpi_cuda> ${HEAT_K} ${HEAT_N}\n    DEPENDS heat_mpi_cuda\n    USES_TERMINAL\n  )\nelse()\n  message(STATUS "No CUDA compiler found: building the MPI version only")\nendif()\n',
    'src/mpi/main.cpp': '#include "heat_mpi.hpp"\n#include <iostream>\n#include <string>\n\nint main(int argc, char* argv[]) {\n    if (argc != 3) {\n        std::cout << "Usage: heat K N\\n";\n        return 1;\n    }\n    run_mpi(std::stoi(argv[1]), std::stoi(argv[2]));\n\n    return 0;\n}\n',
    'src/mpi/heat_mpi.hpp': '#ifndef HEAT_MPI\n#define HEAT_MPI\n\n#pragma once\nvoid run_mpi(int K, int N);\n\n#endif',
    'src/mpi/heat_mpi.cpp': '#include "heat_mpi.hpp"\n#include <mpi.h>\n#include <vector>\n#include <iostream>\n#include <fstream>\n\nvoid exchange_ghosts(std::vector<double>& u, int up, int down, int localRows, int N) {\n    MPI_Sendrecv(&u[1 * N], N, MPI_DOUBLE, up,   0, &u[(localRows + 1) * N], N, MPI_DOUBLE, down, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n    MPI_Sendrecv(&u[localRows * N], N, MPI_DOUBLE, down, 0, &u[0], N, MPI_DOUBLE, up,   0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n}\n\nvoid run_mpi(int K, int N) {\n    MPI_Init(nullptr, nullptr);\n\n    int rank, size;\n    MPI_Comm_rank(MPI_COMM_WORLD, &rank);\n    MPI_Comm_size(MPI_COMM_WORLD, &size);\n\n    if (N % size != 0) {\n        if (rank == 0) std::cerr << "Error: N (" << N << ") must be divisible by the number of processes (" << size << ")\\n";\n        MPI_Finalize();\n        return;\n    }\n\n    int localRows = N / size;\n    int up   = (rank == 0)        ? MPI_PROC_NULL : rank - 1;\n    int down = (rank == size - 1) ? MPI_PROC_NULL : rank + 1;   \n\n    std::vector<double> u((localRows + 2) * N, 0.0);           \n    std::vector<double> uNew((localRows + 2) * N, 0.0);\n    constexpr double alpha = 0.2;\n\n    if (rank == 0) {\n        for (int j = 0; j < N; j++) u[1 * N + j] = uNew[1 * N + j] = 100.0;\n    }\n\n    MPI_Barrier(MPI_COMM_WORLD);\n    double t0 = MPI_Wtime();\n\n    for (int step = 0; step < K; step++){\n        exchange_ghosts(u, up, down, localRows, N);\n        int iStart = (rank == 0)        ? 2 : 1;\n        int iEnd   = (rank == size - 1) ? localRows - 1 : localRows;\n        for (int i = iStart; i <= iEnd; i++) {\n            for (int j = 1; j < N - 1; j++) {\n                int idx = i*N + j;\n                uNew[idx] = u[idx] + alpha * (u[idx - N] + u[idx + N] + u[idx - 1] + u[idx + 1] - 4*u[idx]);\n            }\n        }\n        std::swap(u, uNew);\n    }\n\n    double localTime = MPI_Wtime() - t0;\n\n    double localSum = 0.0;\n    for (int i = 1; i <= localRows; i++){\n        for (int j = 0; j < N; j++){\n            localSum += u[i*N + j];\n        }\n    }\n\n\n    double total = 0.0;\n    MPI_Reduce(&localSum, &total, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);\n    double maxTime = 0.0;\n    MPI_Reduce(&localTime, &maxTime, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n\n    if (N <= 1024) {                       \n        std::vector<double> full;\n        if (rank == 0) full.resize((size_t)N * N);\n\n        MPI_Gather(&u[1 * N], localRows * N, MPI_DOUBLE,     \n                full.data(), localRows * N, MPI_DOUBLE,   \n                0, MPI_COMM_WORLD);\n\n        if (rank == 0) {\n            std::ofstream f("grid.csv");\n            for (int i = 0; i < N; i++)\n                for (int j = 0; j < N; j++)\n                    f << full[i*N + j] << (j + 1 < N ? "," : "\\n");\n        }\n    }\n\n    if (rank == 0) {\n        std::cout << "Average = " << total / (N * N) << std::endl;\n        std::cout << "Time = " << maxTime << " s" << std::endl;\n    }\n\n    MPI_Finalize();\n}',
    'src/cuda/cuda_check.hpp': '#pragma once\n#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstdlib>\n\n// Abort with file and line if a CUDA runtime call fails.\n#define CUDA_CHECK(call)                                                        \\\n    do {                                                                        \\\n        cudaError_t err_ = (call);                                              \\\n        if (err_ != cudaSuccess) {                                              \\\n            std::fprintf(stderr, "CUDA error %s at %s:%d\\n",                    \\\n                         cudaGetErrorString(err_), __FILE__, __LINE__);         \\\n            std::exit(EXIT_FAILURE);                                            \\\n        }                                                                       \\\n    } while (0)\n',
    'src/cuda/heat_gpu.hpp': '// GPU kernels for the 2D heat equation, callable from plain C++ (implemented in heat_gpu.cu).\n#pragma once\n#include <cuda_runtime.h>\n#include <cstddef>\n\n// Kernels that advance the grid by one time step.\nenum class StepKernel {\n    Naive,    // one thread per point, every value loaded from global memory (through the caches)\n    Shared,   // each block first copies its tile plus a one-point halo into shared memory\n};\n\n// Largest number of time steps the temporal kernel does in one launch.\nconstexpr int kMaxTemporalSteps = 8;\n\n// One time step on rows [rowBegin, rowEnd), columns 1..N-2, of an array with N columns:\n//   uNew = u + alpha * (u_up + u_down + u_left + u_right - 4 u)\n// Rows rowBegin-1 and rowEnd must exist (boundary or ghost rows); they are read, not written.\ntemplate <typename T>\nvoid launch_step_rows(const T* u, T* uNew, int N, int rowBegin, int rowEnd, T alpha,\n                      StepKernel kernel = StepKernel::Naive, cudaStream_t stream = 0);\n\n// S time steps (1 <= S <= kMaxTemporalSteps) on the whole N x N grid in a single launch\n// (temporal blocking): each block keeps its tile in shared memory for all S steps, so the\n// grid is read and written once per S steps instead of once per step.\ntemplate <typename T>\nvoid launch_temporal(const T* u, T* uNew, int N, int S, T alpha, cudaStream_t stream = 0);\n\n// Sum of n values on the GPU, accumulated in double. Blocks until the result is ready.\ntemplate <typename T>\ndouble gpu_sum(const T* x, size_t n, cudaStream_t stream = 0);\n',
    'src/cuda/heat_gpu.cu': '// CUDA kernels for the 2D heat equation. See heat_gpu.hpp for the host-side interface.\n#include "heat_gpu.hpp"\n#include "cuda_check.hpp"\n\n#include <algorithm>\n#include <vector>\n\nnamespace {\n\n// ---------------------------------------------------------------------------------------------\n// One time step per launch\n// ---------------------------------------------------------------------------------------------\n\nconstexpr int kBX = 32;   // threads per block along a row (32 = one warp, so loads are coalesced)\nconstexpr int kBY = 8;    // threads per block across rows\n\n// One thread per grid point. Neighbouring threads load overlapping values; the L1/L2 caches\n// serve most of the repeated loads, so each value comes from DRAM roughly once per step.\ntemplate <typename T>\n__global__ void step_naive(const T* __restrict__ u, T* __restrict__ uNew,\n                           int N, int rowBegin, int rowEnd, T alpha) {\n    const int j = blockIdx.x * blockDim.x + threadIdx.x;\n    const int i = rowBegin + blockIdx.y * blockDim.y + threadIdx.y;\n    if (i >= rowEnd || j < 1 || j >= N - 1) return;\n\n    const size_t idx = (size_t)i * N + j;\n    const T c = u[idx];\n    uNew[idx] = c + alpha * (u[idx - N] + u[idx + N] + u[idx - 1] + u[idx + 1] - T(4) * c);\n}\n\n// The block copies its kBY x kBX tile plus a one-point halo into shared memory, then every\n// thread reads its four neighbours from there. Rows rowBegin-1 .. rowEnd exist in memory.\ntemplate <typename T>\n__global__ void step_shared(const T* __restrict__ u, T* __restrict__ uNew,\n                            int N, int rowBegin, int rowEnd, T alpha) {\n    __shared__ T s[kBY + 2][kBX + 2];\n\n    const int tx = threadIdx.x, ty = threadIdx.y;\n    const int j = blockIdx.x * kBX + tx;\n    const int i = rowBegin + blockIdx.y * kBY + ty;\n\n    if (j < N) {\n        if (i <= rowEnd) s[ty + 1][tx + 1] = u[(size_t)i * N + j];\n        if (ty == 0) s[0][tx + 1] = u[(size_t)(i - 1) * N + j];\n        if (ty == kBY - 1 && i + 1 <= rowEnd) s[kBY + 1][tx + 1] = u[(size_t)(i + 1) * N + j];\n    }\n    if (i <= rowEnd) {\n        if (tx == 0 && j >= 1) s[ty + 1][0] = u[(size_t)i * N + j - 1];\n        if (tx == kBX - 1 && j + 1 < N) s[ty + 1][kBX + 1] = u[(size_t)i * N + j + 1];\n    }\n    __syncthreads();\n\n    if (i < rowEnd && j >= 1 && j < N - 1) {\n        const T c = s[ty + 1][tx + 1];\n        uNew[(size_t)i * N + j] =\n            c + alpha * (s[ty][tx + 1] + s[ty + 2][tx + 1] + s[ty + 1][tx] + s[ty + 1][tx + 2] - T(4) * c);\n    }\n}\n\n// ---------------------------------------------------------------------------------------------\n// Several time steps per launch (temporal blocking)\n// ---------------------------------------------------------------------------------------------\n\nconstexpr int kTX = 64;                    // tile width (columns)\nconstexpr int kTY = 32;                    // tile height (rows)\nconstexpr int kTRows = 4;                  // rows handled by each thread\nconstexpr int kTThreadsY = kTY / kTRows;   // block = kTX x kTThreadsY = 512 threads\n\n// Each block loads a kTY x kTX tile into shared memory and advances it S steps there.\n// After s steps only the points at least s away from the tile edge are still exact (the edge\n// has no neighbours outside the tile), so the block writes back the inner\n// (kTY - 2S) x (kTX - 2S) points. Neighbouring tiles overlap by 2S rows/columns, and the\n// points in the overlap are computed by both blocks: extra arithmetic, but S times less\n// DRAM traffic, which is the bottleneck for this stencil.\ntemplate <typename T>\n__global__ void __launch_bounds__(kTX * kTThreadsY)\nstep_temporal(const T* __restrict__ u, T* __restrict__ uNew, int N, int S, T alpha) {\n    __shared__ T buf[2][kTY][kTX];\n\n    const int outX = kTX - 2 * S;              // points written per block along a row\n    const int outY = kTY - 2 * S;\n    const int j0 = blockIdx.x * outX - S;      // global column of tile column 0\n    const int i0 = blockIdx.y * outY - S;      // global row of tile row 0\n    const int c = threadIdx.x;\n    const int gj = j0 + c;\n\n#pragma unroll\n    for (int k = 0; k < kTRows; k++) {\n        const int r = threadIdx.y + k * kTThreadsY;\n        const int gi = i0 + r;\n        buf[0][r][c] = (gi >= 0 && gi < N && gj >= 0 && gj < N) ? u[(size_t)gi * N + gj] : T(0);\n    }\n    __syncthreads();\n\n    for (int s = 0; s < S; s++) {\n        const T (*src)[kTX] = buf[s & 1];\n        T (*dst)[kTX] = buf[(s + 1) & 1];\n#pragma unroll\n        for (int k = 0; k < kTRows; k++) {\n            const int r = threadIdx.y + k * kTThreadsY;\n            const int gi = i0 + r;\n            T v = src[r][c];\n            // Only interior points of the grid change; the fixed boundary and the points\n            // outside the grid keep their values.\n            if (r >= 1 && r < kTY - 1 && c >= 1 && c < kTX - 1 &&\n                gi >= 1 && gi < N - 1 && gj >= 1 && gj < N - 1) {\n                v = v + alpha * (src[r - 1][c] + src[r + 1][c] + src[r][c - 1] + src[r][c + 1] - T(4) * v);\n            }\n            dst[r][c] = v;\n        }\n        __syncthreads();\n    }\n\n    const T (*fin)[kTX] = buf[S & 1];\n    if (c >= S && c < kTX - S && gj < N) {\n#pragma unroll\n        for (int k = 0; k < kTRows; k++) {\n            const int r = threadIdx.y + k * kTThreadsY;\n            const int gi = i0 + r;\n            if (r >= S && r < kTY - S && gi < N) uNew[(size_t)gi * N + gj] = fin[r][c];\n        }\n    }\n}\n\n// ---------------------------------------------------------------------------------------------\n// Reduction\n// ---------------------------------------------------------------------------------------------\n\nconstexpr int kSumThreads = 256;\nconstexpr int kSumMaxBlocks = 1024;\n\n// Each block sums a strided part of x: per-thread partial sums, then a reduction inside each\n// warp with shuffles, then across the block\'s warps. One partial sum per block.\ntemplate <typename T>\n__global__ void sum_partial(const T* __restrict__ x, size_t n, double* __restrict__ partial) {\n    double v = 0.0;\n    for (size_t k = (size_t)blockIdx.x * blockDim.x + threadIdx.x; k < n; k += (size_t)gridDim.x * blockDim.x)\n        v += (double)x[k];\n\n    for (int offset = 16; offset > 0; offset >>= 1) v += __shfl_down_sync(0xffffffffu, v, offset);\n\n    __shared__ double warpSums[32];\n    const int lane = threadIdx.x & 31;\n    const int warp = threadIdx.x >> 5;\n    if (lane == 0) warpSums[warp] = v;\n    __syncthreads();\n\n    if (warp == 0) {\n        v = (lane < (int)(blockDim.x >> 5)) ? warpSums[lane] : 0.0;\n        for (int offset = 16; offset > 0; offset >>= 1) v += __shfl_down_sync(0xffffffffu, v, offset);\n        if (lane == 0) partial[blockIdx.x] = v;\n    }\n}\n\n}  // namespace\n\n// ---------------------------------------------------------------------------------------------\n// Host-side launchers\n// ---------------------------------------------------------------------------------------------\n\ntemplate <typename T>\nvoid launch_step_rows(const T* u, T* uNew, int N, int rowBegin, int rowEnd, T alpha,\n                      StepKernel kernel, cudaStream_t stream) {\n    if (rowEnd <= rowBegin) return;\n    const dim3 block(kBX, kBY);\n    const dim3 grid((N + kBX - 1) / kBX, (rowEnd - rowBegin + kBY - 1) / kBY);\n    if (kernel == StepKernel::Shared)\n        step_shared<T><<<grid, block, 0, stream>>>(u, uNew, N, rowBegin, rowEnd, alpha);\n    else\n        step_naive<T><<<grid, block, 0, stream>>>(u, uNew, N, rowBegin, rowEnd, alpha);\n    CUDA_CHECK(cudaGetLastError());\n}\n\ntemplate <typename T>\nvoid launch_temporal(const T* u, T* uNew, int N, int S, T alpha, cudaStream_t stream) {\n    if (S < 1 || S > kMaxTemporalSteps) {\n        std::fprintf(stderr, "launch_temporal: S must be between 1 and %d\\n", kMaxTemporalSteps);\n        std::exit(EXIT_FAILURE);\n    }\n    const int outX = kTX - 2 * S, outY = kTY - 2 * S;\n    const dim3 block(kTX, kTThreadsY);\n    const dim3 grid((N + outX - 1) / outX, (N + outY - 1) / outY);\n    step_temporal<T><<<grid, block, 0, stream>>>(u, uNew, N, S, alpha);\n    CUDA_CHECK(cudaGetLastError());\n}\n\ntemplate <typename T>\ndouble gpu_sum(const T* x, size_t n, cudaStream_t stream) {\n    if (n == 0) return 0.0;\n    const int blocks = (int)std::min<size_t>(kSumMaxBlocks, (n + kSumThreads - 1) / kSumThreads);\n\n    double* dPartial = nullptr;\n    CUDA_CHECK(cudaMalloc(&dPartial, blocks * sizeof(double)));\n    sum_partial<T><<<blocks, kSumThreads, 0, stream>>>(x, n, dPartial);\n    CUDA_CHECK(cudaGetLastError());\n\n    std::vector<double> partial(blocks);\n    CUDA_CHECK(cudaMemcpyAsync(partial.data(), dPartial, blocks * sizeof(double), cudaMemcpyDeviceToHost, stream));\n    CUDA_CHECK(cudaStreamSynchronize(stream));\n    CUDA_CHECK(cudaFree(dPartial));\n\n    double total = 0.0;\n    for (double p : partial) total += p;   // fixed order, so the result is reproducible\n    return total;\n}\n\ntemplate void launch_step_rows<float>(const float*, float*, int, int, int, float, StepKernel, cudaStream_t);\ntemplate void launch_step_rows<double>(const double*, double*, int, int, int, double, StepKernel, cudaStream_t);\ntemplate void launch_temporal<float>(const float*, float*, int, int, float, cudaStream_t);\ntemplate void launch_temporal<double>(const double*, double*, int, int, double, cudaStream_t);\ntemplate double gpu_sum<float>(const float*, size_t, cudaStream_t);\ntemplate double gpu_sum<double>(const double*, size_t, cudaStream_t);\n',
    'src/cuda/heat_reference.hpp': '// Serial CPU version of the scheme, used to check the GPU results point by point.\n#pragma once\n#include <cstddef>\n#include <utility>\n#include <vector>\n\ninline std::vector<double> heat_reference(int K, int N, double alpha = 0.2) {\n    std::vector<double> u((size_t)N * N, 0.0);\n    for (int j = 0; j < N; j++) u[j] = 100.0;\n    std::vector<double> uNew = u;\n\n    for (int step = 0; step < K; step++) {\n        for (int i = 1; i < N - 1; i++) {\n            for (int j = 1; j < N - 1; j++) {\n                const size_t idx = (size_t)i * N + j;\n                uNew[idx] = u[idx] + alpha * (u[idx - N] + u[idx + N] + u[idx - 1] + u[idx + 1] - 4.0 * u[idx]);\n            }\n        }\n        std::swap(u, uNew);\n    }\n    return u;\n}\n',
    'src/cuda/heat_cuda.cpp': '// Single-GPU 2D heat solver: argument parsing, timing and output. The kernels are in heat_gpu.cu.\n#include "cuda_check.hpp"\n#include "heat_gpu.hpp"\n#include "heat_reference.hpp"\n\n#include <algorithm>\n#include <cmath>\n#include <fstream>\n#include <iostream>\n#include <string>\n#include <utility>\n#include <vector>\n\nnamespace {\n\nstruct Options {\n    int K = 0;\n    int N = 0;\n    std::string kernel = "naive";      // naive | shared | temporal\n    std::string precision = "double";  // double | float\n    int tsteps = 4;                    // time steps per launch for the temporal kernel\n    bool verify = false;               // compare every grid point with a CPU run\n    bool csv = true;                   // write grid.csv when N <= 1024\n};\n\nvoid usage() {\n    std::cout << "Usage: heat_cuda K N [options]\\n"\n                 "  K                 number of time steps\\n"\n                 "  N                 grid size (N x N, N >= 3)\\n"\n                 "  --kernel NAME     naive (default), shared or temporal\\n"\n                 "  --precision P     double (default) or float\\n"\n                 "  --tsteps S        time steps per launch for --kernel temporal (1-"\n              << kMaxTemporalSteps << ", default 4)\\n"\n                 "  --verify          compare every grid point with a CPU run of the same problem\\n"\n                 "  --no-csv          do not write grid.csv\\n";\n}\n\nbool parse_args(int argc, char** argv, Options& o) {\n    if (argc < 3) return false;\n    try {\n        o.K = std::stoi(argv[1]);\n        o.N = std::stoi(argv[2]);\n        for (int a = 3; a < argc; a++) {\n            const std::string arg = argv[a];\n            const bool hasValue = a + 1 < argc;\n            if (arg == "--kernel" && hasValue) o.kernel = argv[++a];\n            else if (arg == "--precision" && hasValue) o.precision = argv[++a];\n            else if (arg == "--tsteps" && hasValue) o.tsteps = std::stoi(argv[++a]);\n            else if (arg == "--verify") o.verify = true;\n            else if (arg == "--no-csv") o.csv = false;\n            else return false;\n        }\n    } catch (...) {\n        return false;\n    }\n    const bool kernelOk = o.kernel == "naive" || o.kernel == "shared" || o.kernel == "temporal";\n    const bool precisionOk = o.precision == "double" || o.precision == "float";\n    return o.K >= 0 && o.N >= 3 && kernelOk && precisionOk && o.tsteps >= 1 && o.tsteps <= kMaxTemporalSteps;\n}\n\n// Theoretical peak memory bandwidth in GB/s, or 0 if the driver does not report it.\ndouble peak_bandwidth_gbs(int device) {\n    int clockKHz = 0, busBits = 0;\n    if (cudaDeviceGetAttribute(&clockKHz, cudaDevAttrMemoryClockRate, device) != cudaSuccess ||\n        cudaDeviceGetAttribute(&busBits, cudaDevAttrGlobalMemoryBusWidth, device) != cudaSuccess) {\n        cudaGetLastError();   // clear the error\n        return 0.0;\n    }\n    return 2.0 * clockKHz * 1e3 * (busBits / 8.0) / 1e9;   // two transfers per clock (DDR)\n}\n\n// Advance the grid by `steps` time steps. dU always points to the newest grid afterwards.\ntemplate <typename T>\nvoid advance(T*& dU, T*& dUNew, int N, int steps, T alpha, const Options& o) {\n    if (o.kernel == "temporal") {\n        for (int done = 0; done < steps;) {\n            const int s = std::min(o.tsteps, steps - done);   // the last launch may do fewer steps\n            launch_temporal(dU, dUNew, N, s, alpha);\n            std::swap(dU, dUNew);\n            done += s;\n        }\n    } else {\n        const StepKernel kernel = (o.kernel == "shared") ? StepKernel::Shared : StepKernel::Naive;\n        for (int step = 0; step < steps; step++) {\n            launch_step_rows(dU, dUNew, N, 1, N - 1, alpha, kernel);\n            std::swap(dU, dUNew);\n        }\n    }\n}\n\ntemplate <typename T>\nint run(const Options& o, double peakGBs) {\n    const int K = o.K, N = o.N;\n    const size_t count = (size_t)N * N;\n    const size_t bytes = count * sizeof(T);\n    const T alpha = T(0.2);\n\n    // Initial condition: top row = 100, everything else 0.\n    std::vector<T> h(count, T(0));\n    for (int j = 0; j < N; j++) h[j] = T(100);\n\n    T *dU = nullptr, *dUNew = nullptr;\n    CUDA_CHECK(cudaMalloc(&dU, bytes));\n    CUDA_CHECK(cudaMalloc(&dUNew, bytes));\n\n    // Untimed warm-up step: the first launch of a kernel also loads it onto the GPU.\n    CUDA_CHECK(cudaMemcpy(dU, h.data(), bytes, cudaMemcpyHostToDevice));\n    CUDA_CHECK(cudaMemcpy(dUNew, h.data(), bytes, cudaMemcpyHostToDevice));\n    advance(dU, dUNew, N, 1, alpha, o);\n    CUDA_CHECK(cudaDeviceSynchronize());\n    CUDA_CHECK(cudaMemcpy(dU, h.data(), bytes, cudaMemcpyHostToDevice));\n    CUDA_CHECK(cudaMemcpy(dUNew, h.data(), bytes, cudaMemcpyHostToDevice));\n\n    cudaEvent_t start, stop;\n    CUDA_CHECK(cudaEventCreate(&start));\n    CUDA_CHECK(cudaEventCreate(&stop));\n\n    CUDA_CHECK(cudaEventRecord(start));\n    advance(dU, dUNew, N, K, alpha, o);\n    CUDA_CHECK(cudaEventRecord(stop));\n    CUDA_CHECK(cudaEventSynchronize(stop));\n\n    float ms = 0.0f;\n    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));\n    const double seconds = ms / 1000.0;\n\n    const double average = gpu_sum(dU, count) / ((double)N * N);\n\n    const bool writeCsv = o.csv && N <= 1024;\n    if (writeCsv || o.verify) CUDA_CHECK(cudaMemcpy(h.data(), dU, bytes, cudaMemcpyDeviceToHost));\n\n    if (writeCsv) {\n        std::ofstream f("grid.csv");\n        for (int i = 0; i < N; i++)\n            for (int j = 0; j < N; j++)\n                f << h[(size_t)i * N + j] << (j + 1 < N ? "," : "\\n");\n    }\n\n    std::string label = o.kernel;\n    if (o.kernel == "temporal") label += " (S=" + std::to_string(o.tsteps) + ")";\n    std::cout << "Kernel: " << label << ", precision: " << o.precision << "\\n";\n    std::cout << "Average = " << average << "\\n";\n    std::cout << "Time = " << seconds << " s\\n";\n    if (seconds > 0) {\n        // Lattice updates per second, and the DRAM traffic a one-step-per-launch kernel would\n        // need for that rate (one read + one write per point per step).\n        const double glups = (double)N * N * K / seconds / 1e9;\n        std::cout << "Throughput = " << glups << " GLUP/s\\n";\n        std::cout << "Bandwidth = " << glups * 2 * sizeof(T) << " GB/s effective";\n        if (peakGBs > 0) std::cout << " (device peak " << peakGBs << " GB/s)";\n        std::cout << "\\n";\n    }\n\n    int status = 0;\n    if (o.verify) {\n        const std::vector<double> ref = heat_reference(K, N);\n        double maxDiff = 0.0;\n        for (size_t k = 0; k < count; k++) {\n            const double d = std::fabs((double)h[k] - ref[k]);\n            if (!(d <= maxDiff)) maxDiff = d;   // also catches NaN\n        }\n        const double tol = (sizeof(T) == sizeof(double)) ? 1e-9 : 1e-2;\n        const bool ok = maxDiff <= tol;\n        std::cout << "Verify: max |gpu - cpu| = " << maxDiff << " (tolerance " << tol << ") "\n                  << (ok ? "OK" : "FAILED") << "\\n";\n        status = ok ? 0 : 1;\n    }\n\n    CUDA_CHECK(cudaEventDestroy(start));\n    CUDA_CHECK(cudaEventDestroy(stop));\n    CUDA_CHECK(cudaFree(dU));\n    CUDA_CHECK(cudaFree(dUNew));\n    return status;\n}\n\n}  // namespace\n\nint main(int argc, char** argv) {\n    Options o;\n    if (!parse_args(argc, argv, o)) {\n        usage();\n        return 1;\n    }\n\n    int device = 0;\n    cudaDeviceProp prop;\n    CUDA_CHECK(cudaGetDevice(&device));\n    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));\n    std::cout << "GPU: " << prop.name << "\\n";\n\n    const double peak = peak_bandwidth_gbs(device);\n    return (o.precision == "float") ? run<float>(o, peak) : run<double>(o, peak);\n}\n',
    'src/cuda/heat_mpi_cuda.cpp': '// Multi-GPU 2D heat solver: MPI between processes, CUDA inside each process.\n// Same row decomposition as heat_mpi.cpp, but each rank keeps its strip (plus two ghost rows)\n// in GPU memory. The kernels are in heat_gpu.cu.\n#include "cuda_check.hpp"\n#include "heat_gpu.hpp"\n\n#include <mpi.h>\n\n#include <algorithm>\n#include <fstream>\n#include <iostream>\n#include <string>\n#include <utility>\n#include <vector>\n\nint main(int argc, char** argv) {\n    MPI_Init(&argc, &argv);\n\n    int rank = 0, size = 1;\n    MPI_Comm_rank(MPI_COMM_WORLD, &rank);\n    MPI_Comm_size(MPI_COMM_WORLD, &size);\n\n    // ---- Arguments ----\n    int K = 0, N = 0;\n    bool overlap = true;     // compute interior rows while the halo exchange runs\n    bool cudaAware = false;  // pass GPU pointers straight to MPI (needs a CUDA-aware MPI build)\n    bool argsOk = argc >= 3;\n    try {\n        if (argsOk) {\n            K = std::stoi(argv[1]);\n            N = std::stoi(argv[2]);\n        }\n    } catch (...) {\n        argsOk = false;\n    }\n    for (int a = 3; argsOk && a < argc; a++) {\n        const std::string arg = argv[a];\n        if (arg == "--no-overlap") overlap = false;\n        else if (arg == "--cuda-aware") cudaAware = true;\n        else argsOk = false;\n    }\n    if (!argsOk || K < 0 || N < 3) {\n        if (rank == 0) std::cout << "Usage: heat_mpi_cuda K N [--no-overlap] [--cuda-aware]\\n";\n        MPI_Finalize();\n        return 1;\n    }\n    if (N % size != 0) {\n        if (rank == 0)\n            std::cerr << "Error: N (" << N << ") must be divisible by the number of processes (" << size << ")\\n";\n        MPI_Finalize();\n        return 1;\n    }\n\n    // ---- One GPU per rank on each node (ranks share GPUs round-robin if there are fewer) ----\n    MPI_Comm nodeComm;\n    MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, rank, MPI_INFO_NULL, &nodeComm);\n    int nodeRank = 0;\n    MPI_Comm_rank(nodeComm, &nodeRank);\n    MPI_Comm_free(&nodeComm);\n\n    int deviceCount = 0;\n    CUDA_CHECK(cudaGetDeviceCount(&deviceCount));\n    CUDA_CHECK(cudaSetDevice(nodeRank % deviceCount));\n\n    // ---- Local strip: rows 1..localRows, ghost rows 0 and localRows+1 ----\n    const int localRows = N / size;\n    const int up = (rank == 0) ? MPI_PROC_NULL : rank - 1;\n    const int down = (rank == size - 1) ? MPI_PROC_NULL : rank + 1;\n    const size_t count = (size_t)(localRows + 2) * N;\n    const size_t bytes = count * sizeof(double);\n    const size_t rowBytes = (size_t)N * sizeof(double);\n    constexpr double alpha = 0.2;\n\n    std::vector<double> h(count, 0.0);\n    if (rank == 0)\n        for (int j = 0; j < N; j++) h[(size_t)N + j] = 100.0;   // global row 0 is local row 1 of rank 0\n\n    double *dU = nullptr, *dUNew = nullptr;\n    CUDA_CHECK(cudaMalloc(&dU, bytes));\n    CUDA_CHECK(cudaMalloc(&dUNew, bytes));\n\n    // Pinned host buffers for the host-staged halo exchange (pinned memory allows async copies).\n    double *sendUp = nullptr, *sendDown = nullptr, *recvUp = nullptr, *recvDown = nullptr;\n    CUDA_CHECK(cudaMallocHost(&sendUp, rowBytes));\n    CUDA_CHECK(cudaMallocHost(&sendDown, rowBytes));\n    CUDA_CHECK(cudaMallocHost(&recvUp, rowBytes));\n    CUDA_CHECK(cudaMallocHost(&recvDown, rowBytes));\n\n    cudaStream_t compute, comm;\n    CUDA_CHECK(cudaStreamCreateWithFlags(&compute, cudaStreamNonBlocking));\n    CUDA_CHECK(cudaStreamCreateWithFlags(&comm, cudaStreamNonBlocking));\n\n    // Rows this rank updates (local indices, inclusive), as in heat_mpi.cpp.\n    const int iStart = (rank == 0) ? 2 : 1;\n    const int iEnd = (rank == size - 1) ? localRows - 1 : localRows;\n    // Rows 1 and localRows read a ghost row; the rows between them do not.\n    const int innerBegin = std::max(iStart, 2);\n    const int innerEnd = std::min(iEnd, localRows - 1) + 1;   // exclusive\n    std::vector<int> edgeRows;\n    for (int r : {1, localRows})\n        if (r >= iStart && r <= iEnd && std::find(edgeRows.begin(), edgeRows.end(), r) == edgeRows.end())\n            edgeRows.push_back(r);\n\n    // Fill ghost rows 0 and localRows+1 of d with the neighbours\' edge rows.\n    // Device work is queued on the comm stream.\n    auto exchange = [&](double* d) {\n        double* firstRow = d + N;\n        double* lastRow = d + (size_t)localRows * N;\n        double* ghostTop = d;\n        double* ghostBottom = d + (size_t)(localRows + 1) * N;\n\n        if (cudaAware) {\n            MPI_Sendrecv(firstRow, N, MPI_DOUBLE, up, 0, ghostBottom, N, MPI_DOUBLE, down, 0,\n                         MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n            MPI_Sendrecv(lastRow, N, MPI_DOUBLE, down, 1, ghostTop, N, MPI_DOUBLE, up, 1,\n                         MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n            return;\n        }\n\n        // Host-staged: GPU -> pinned buffer -> MPI -> pinned buffer -> GPU.\n        if (up != MPI_PROC_NULL)\n            CUDA_CHECK(cudaMemcpyAsync(sendUp, firstRow, rowBytes, cudaMemcpyDeviceToHost, comm));\n        if (down != MPI_PROC_NULL)\n            CUDA_CHECK(cudaMemcpyAsync(sendDown, lastRow, rowBytes, cudaMemcpyDeviceToHost, comm));\n        CUDA_CHECK(cudaStreamSynchronize(comm));\n\n        MPI_Sendrecv(sendUp, N, MPI_DOUBLE, up, 0, recvDown, N, MPI_DOUBLE, down, 0,\n                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n        MPI_Sendrecv(sendDown, N, MPI_DOUBLE, down, 1, recvUp, N, MPI_DOUBLE, up, 1,\n                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n\n        if (up != MPI_PROC_NULL)\n            CUDA_CHECK(cudaMemcpyAsync(ghostTop, recvUp, rowBytes, cudaMemcpyHostToDevice, comm));\n        if (down != MPI_PROC_NULL)\n            CUDA_CHECK(cudaMemcpyAsync(ghostBottom, recvDown, rowBytes, cudaMemcpyHostToDevice, comm));\n    };\n\n    auto step = [&]() {\n        if (overlap) {\n            // The interior kernel runs on its own stream while the edge rows travel through MPI;\n            // the two edge rows are updated once their ghost rows have arrived.\n            launch_step_rows(dU, dUNew, N, innerBegin, innerEnd, alpha, StepKernel::Naive, compute);\n            exchange(dU);\n            for (int r : edgeRows) launch_step_rows(dU, dUNew, N, r, r + 1, alpha, StepKernel::Naive, comm);\n        } else {\n            exchange(dU);\n            launch_step_rows(dU, dUNew, N, iStart, iEnd + 1, alpha, StepKernel::Naive, comm);\n        }\n        CUDA_CHECK(cudaStreamSynchronize(compute));\n        CUDA_CHECK(cudaStreamSynchronize(comm));\n        std::swap(dU, dUNew);\n    };\n\n    // Untimed warm-up step (loads the kernels onto the GPU), then reset to the initial condition.\n    CUDA_CHECK(cudaMemcpy(dU, h.data(), bytes, cudaMemcpyHostToDevice));\n    CUDA_CHECK(cudaMemcpy(dUNew, h.data(), bytes, cudaMemcpyHostToDevice));\n    step();\n    CUDA_CHECK(cudaMemcpy(dU, h.data(), bytes, cudaMemcpyHostToDevice));\n    CUDA_CHECK(cudaMemcpy(dUNew, h.data(), bytes, cudaMemcpyHostToDevice));\n\n    MPI_Barrier(MPI_COMM_WORLD);\n    const double t0 = MPI_Wtime();\n    for (int s = 0; s < K; s++) step();\n    const double localTime = MPI_Wtime() - t0;\n\n    // ---- Results ----\n    const double localSum = gpu_sum(dU + N, (size_t)localRows * N);\n    double total = 0.0, maxTime = 0.0;\n    MPI_Reduce(&localSum, &total, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);\n    MPI_Reduce(&localTime, &maxTime, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n\n    if (N <= 1024) {\n        std::vector<double> mine((size_t)localRows * N);\n        CUDA_CHECK(cudaMemcpy(mine.data(), dU + N, mine.size() * sizeof(double), cudaMemcpyDeviceToHost));\n        std::vector<double> full;\n        if (rank == 0) full.resize((size_t)N * N);\n        MPI_Gather(mine.data(), localRows * N, MPI_DOUBLE, full.data(), localRows * N, MPI_DOUBLE, 0, MPI_COMM_WORLD);\n        if (rank == 0) {\n            std::ofstream f("grid.csv");\n            for (int i = 0; i < N; i++)\n                for (int j = 0; j < N; j++)\n                    f << full[(size_t)i * N + j] << (j + 1 < N ? "," : "\\n");\n        }\n    }\n\n    if (rank == 0) {\n        std::cout << "Ranks = " << size << ", GPUs on rank 0\'s node = " << deviceCount\n                  << ", overlap = " << (overlap ? "on" : "off")\n                  << ", halo transport = " << (cudaAware ? "CUDA-aware MPI" : "host-staged") << "\\n";\n        std::cout << "Average = " << total / ((double)N * N) << "\\n";\n        std::cout << "Time = " << maxTime << " s\\n";\n    }\n\n    CUDA_CHECK(cudaStreamDestroy(compute));\n    CUDA_CHECK(cudaStreamDestroy(comm));\n    CUDA_CHECK(cudaFreeHost(sendUp));\n    CUDA_CHECK(cudaFreeHost(sendDown));\n    CUDA_CHECK(cudaFreeHost(recvUp));\n    CUDA_CHECK(cudaFreeHost(recvDown));\n    CUDA_CHECK(cudaFree(dU));\n    CUDA_CHECK(cudaFree(dUNew));\n\n    MPI_Finalize();\n    return 0;\n}\n',
    'scripts/test_cuda.sh': '#!/usr/bin/env bash\n# Correctness checks for the CUDA programs (needs an NVIDIA GPU; build first with CMake).\n# Usage: ./scripts/test_cuda.sh\ncd "$(dirname "$0")/.."\nBIN=${BIN:-./build}\nEXPECTED=6.01949\nCORES=$(nproc 2>/dev/null || sysctl -n hw.ncpu)\nfailures=0\n\nreport() {  # report <exit status> <text>\n    if [ "$1" -eq 0 ]; then echo "  OK      $2"; else echo "  FAILED  $2"; failures=$((failures + 1)); fi\n}\n\nmpi() {  # mpi <np> <command...>: run with mpirun, oversubscribing if there are fewer cores\n    local np=$1 extra=""\n    shift\n    if [ "$np" -gt "$CORES" ]; then extra="--oversubscribe"; fi\n    mpirun $extra -np "$np" "$@"\n}\n\necho "== heat_cuda: average for N=256, K=1000 (expected $EXPECTED) =="\nfor kernel in naive shared temporal; do\n    avg=$("$BIN/heat_cuda" 1000 256 --kernel "$kernel" --no-csv | awk \'$1 == "Average" {print $3}\')\n    [ "$avg" = "$EXPECTED" ]\n    report $? "--kernel $kernel: Average = $avg"\ndone\n\n# Every grid point against a CPU run. N is not a multiple of any tile size, and K is not a\n# multiple of the temporal step counts. K=5000, N=150 lets the heat reach the whole grid, so a\n# mistake anywhere shows up (with few steps most of the grid is still exactly 0).\necho\necho "== heat_cuda: every grid point vs the CPU =="\nfor problem in "257 300" "5000 150"; do\n    for precision in double float; do\n        for args in "--kernel naive" "--kernel shared" "--kernel temporal --tsteps 1" \\\n                    "--kernel temporal --tsteps 3" "--kernel temporal --tsteps 8"; do\n            line=$("$BIN/heat_cuda" $problem $args --precision $precision --verify --no-csv | grep \'^Verify\')\n            echo "$line" | grep -q \' OK$\'\n            report $? "K N = $problem, $args --precision $precision: ${line#Verify: }"\n        done\n    done\ndone\n\nif [ -x "$BIN/heat_mpi_cuda" ] && command -v mpirun > /dev/null; then\n    echo\n    echo "== heat_mpi_cuda: average for N=256, K=1000 (expected $EXPECTED; ranks may share one GPU) =="\n    for np in 1 2 4; do\n        for mode in "" "--no-overlap"; do\n            avg=$(mpi "$np" "$BIN/heat_mpi_cuda" 1000 256 $mode | awk \'$1 == "Average" {print $3}\')\n            [ "$avg" = "$EXPECTED" ]\n            report $? "np=$np ${mode:-overlap}: Average = $avg"\n        done\n    done\nfi\n\necho\nif [ "$failures" -eq 0 ]; then\n    echo "All CUDA checks passed"\nelse\n    echo "$failures CUDA check(s) FAILED"\n    exit 1\nfi\n',
    'scripts/bench_cuda.sh': '#!/usr/bin/env bash\n# Benchmark the CUDA programs and write the tables to results/results_cuda.md (needs an NVIDIA GPU).\n# Usage: ./scripts/bench_cuda.sh                       (defaults)\n#        SIZES="4096 8192" K=100 ./scripts/bench_cuda.sh\nset -e\ncd "$(dirname "$0")/.."\nBIN=${BIN:-./build}\nSIZES=${SIZES:-"4096 8192 16384"}\nK=${K:-200}\nTSTEPS=${TSTEPS:-4}       # time steps per launch for the temporal kernel in the main table\nSWEEP_N=${SWEEP_N:-8192}  # grid size for the temporal-steps sweep and the MPI + CUDA runs\nOUT=results/results_cuda.md\nCORES=$(nproc 2>/dev/null || sysctl -n hw.ncpu)\n\nvalue() {  # value <key> : print the number after "<key> =" in the program output on stdin\n    awk -v key="$1" \'$1 == key {print $3}\'\n}\nratio() { awk -v a="$1" -v b="$2" \'BEGIN {printf "%.2f", a / b}\'; }\nfmt() { awk -v x="$1" \'BEGIN {printf "%.4g", x}\'; }\n\ngpu=$("$BIN/heat_cuda" 1 16 --no-csv | sed -n \'s/^GPU: //p\')\npeak=$("$BIN/heat_cuda" 1 16 --no-csv | sed -n \'s/.*device peak \\([0-9.]*\\) GB.*/\\1/p\')\n\n{\n    echo "GPU: $gpu (peak memory bandwidth ${peak:-?} GB/s), K = $K time steps"\n    echo\n    echo "### Kernels"\n    echo\n    echo "| N | precision | kernel | time (s) | GLUP/s | effective GB/s | speedup vs naive |"\n    echo "|---|---|---|---|---|---|---|"\n} > "$OUT"\n\necho "== Kernels (K=$K) =="\nfor N in $SIZES; do\n    for precision in double float; do\n        base=""\n        for kernel in naive shared temporal; do\n            out=$("$BIN/heat_cuda" "$K" "$N" --kernel "$kernel" --precision "$precision" --tsteps "$TSTEPS" --no-csv)\n            t=$(echo "$out" | value Time)\n            g=$(echo "$out" | value Throughput)\n            bw=$(echo "$out" | value Bandwidth)\n            if [ -z "$base" ]; then base=$t; fi\n            label=$kernel\n            if [ "$kernel" = temporal ]; then label="temporal (S=$TSTEPS)"; fi\n            row="| $N | $precision | $label | $(fmt "$t") | $(fmt "$g") | $(fmt "$bw") | $(ratio "$base" "$t") |"\n            echo "$row"\n            echo "$row" >> "$OUT"\n        done\n    done\ndone\n\n{\n    echo\n    echo "### Temporal blocking: time steps per launch (N = $SWEEP_N)"\n    echo\n    echo "| S | double: time (s) | double: GLUP/s | float: time (s) | float: GLUP/s |"\n    echo "|---|---|---|---|---|"\n} >> "$OUT"\n\necho\necho "== Temporal blocking sweep (N=$SWEEP_N) =="\nfor S in 1 2 3 4 5 6 7 8; do\n    row="| $S |"\n    for precision in double float; do\n        out=$("$BIN/heat_cuda" "$K" "$SWEEP_N" --kernel temporal --tsteps "$S" --precision "$precision" --no-csv)\n        row="$row $(fmt "$(echo "$out" | value Time)") | $(fmt "$(echo "$out" | value Throughput)") |"\n    done\n    echo "$row"\n    echo "$row" >> "$OUT"\ndone\n\nif [ -x "$BIN/heat_mpi_cuda" ] && command -v mpirun > /dev/null; then\n    gpus=$(nvidia-smi -L 2>/dev/null | wc -l | tr -d \' \')\n    {\n        echo\n        echo "### MPI + CUDA (N = $SWEEP_N, double, naive kernel, $gpus GPU(s); ranks share GPUs round-robin)"\n        echo\n        echo "| ranks | overlap | time (s) |"\n        echo "|---|---|---|"\n    } >> "$OUT"\n    echo\n    echo "== MPI + CUDA (N=$SWEEP_N, $gpus GPU(s)) =="\n    for np in 1 2 4; do\n        extra=""\n        if [ "$np" -gt "$CORES" ]; then extra="--oversubscribe"; fi\n        for mode in on off; do\n            flag=""\n            if [ "$mode" = off ]; then flag="--no-overlap"; fi\n            t=$(mpirun $extra -np "$np" "$BIN/heat_mpi_cuda" "$K" "$SWEEP_N" $flag | value Time)\n            row="| $np | $mode | $(fmt "$t") |"\n            echo "$row"\n            echo "$row" >> "$OUT"\n        done\n    done\nfi\n\necho\necho "Tables saved to $OUT"\n',
}

os.makedirs("/content/heat", exist_ok=True)
os.chdir("/content/heat")
os.makedirs("results", exist_ok=True)
for name, content in FILES.items():
    os.makedirs(os.path.dirname(name) or '.', exist_ok=True)
    with open(name, 'w') as f:
        f.write(content)
    if name.endswith('.sh'):
        os.chmod(name, 0o755)
print("Wrote", len(FILES), "files to", os.getcwd())

## Tools: GPU, CUDA, CMake, MPI

In [ ]:
import os, shutil
# Colab runs as root; Open MPI refuses that unless told it is fine.
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
if shutil.which("mpicxx") is None:
    !apt-get -qq update > /dev/null && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!nvcc --version | tail -2
!cmake --version | head -1
!mpirun --version | head -1

## Build

In [ ]:
cap = !nvidia-smi --query-gpu=compute_cap --format=csv,noheader
arch = cap[0].strip().replace(".", "")   # e.g. 7.5 -> 75 for a T4
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={arch} 2>&1 | tail -3
!cmake --build build -j 4 2>&1 | grep -E "error|warning|Built target"

## Correctness
Every kernel, in double and float, is compared point by point with a CPU run.

In [ ]:
!./scripts/test_cuda.sh

## Benchmark
Takes a minute or two.

In [ ]:
!./scripts/bench_cuda.sh > /dev/null && cat results/results_cuda.md

## Plot

In [ ]:
!./build/heat_cuda 1000 256 --kernel temporal
import numpy as np
import matplotlib.pyplot as plt
grid = np.loadtxt("grid.csv", delimiter=",")
plt.imshow(grid, cmap="hot")
plt.colorbar(label="temperature")
plt.title("2D heat equation on the GPU (top edge = 100)")
plt.show()

## Download the results (optional)

In [ ]:
from google.colab import files
files.download("results/results_cuda.md")